In [6]:
from bbs_iss.entities.holder import HolderInstance
from bbs_iss.entities.issuer import IssuerInstance
import bbs_iss.utils.utils as utils
import bbs_iss.interfaces.requests_api as api
import os
issuer = IssuerInstance()
pub_key = issuer.public_key
holder = HolderInstance()
attributes = api.IssuanceAttributes()
attributes.append("secret", utils.gen_link_secret(), api.AttributeType.HIDDEN)
attributes.append("not_secret", "very not secret", api.AttributeType.REVEALED)

# Request to Issuer
init_request = holder.issuance_request(issuer_pub_key=pub_key, attributes=attributes, cred_name="test-cred")

# Process Request -> send Freshness Response
freshness_response = issuer.process_request(init_request)

# Process Freshness Response -> send Blind Sign Request
blind_sign_request = holder.process_request(freshness_response)

# Process Blind Sign Response -> send Forward VC Response
forward_vc_response = issuer.process_request(blind_sign_request)

# Verify VC
holder.process_request(forward_vc_response)

True

In [7]:
holder.credentials

{'test-cred': <bbs_iss.interfaces.credential.VerifiableCredential at 0x7f3b56db2650>}

In [8]:
holder.credentials['test-cred'].to_dict()

{'@context': ['https://www.w3.org/2018/credentials/v1',
  'https://w3id.org/security/bbs/v1'],
 'type': ['VerifiableCredential', 'MockCredential'],
 'issuer': 'Mock-Issuer',
 'credentialSubject': {'secret': 'a19e7265d3fe777d46560a6d762c3d76a6c4643c10fa4c12ff9507f61715d797',
  'not_secret': 'very not secret',
  'metaHash': '35cf5a1e31b17f86eaa956106dd9de60931f106d02337074357f95264dedfe52'},
 'proof': 'ab82a4d8805028f7a61b905f5a6c1fda6935228b6f0c3647b68fc874e51b9d7eaf0f55a4fca6bec650c4335b0df164ca09290ec42cfc704df4efc9528b92319f005103c781b497302a4f639eb2cab3190d64e971f84b5fa4d24be6df5df43a8f1a63e5fe1ea68705d5f0e6398db350f0'}

In [9]:
holder.credentials['test-cred'].issuer = "Changed-Issuer"
holder.credentials['test-cred'].to_dict()

{'@context': ['https://www.w3.org/2018/credentials/v1',
  'https://w3id.org/security/bbs/v1'],
 'type': ['VerifiableCredential', 'MockCredential'],
 'issuer': 'Changed-Issuer',
 'credentialSubject': {'secret': 'a19e7265d3fe777d46560a6d762c3d76a6c4643c10fa4c12ff9507f61715d797',
  'not_secret': 'very not secret',
  'metaHash': '35cf5a1e31b17f86eaa956106dd9de60931f106d02337074357f95264dedfe52'},
 'proof': 'ab82a4d8805028f7a61b905f5a6c1fda6935228b6f0c3647b68fc874e51b9d7eaf0f55a4fca6bec650c4335b0df164ca09290ec42cfc704df4efc9528b92319f005103c781b497302a4f639eb2cab3190d64e971f84b5fa4d24be6df5df43a8f1a63e5fe1ea68705d5f0e6398db350f0'}

In [10]:
holder.verify_vc(pub_key=pub_key, vc_name='test-cred')

False